# 12 — The naive phase embedding tracks $K$

You hold a test bed with a known answer and two classical yardsticks. Now we make the first move of the capstone proper: carry the dyad's phases across the bridge from `10`, turn each phase into a **quantum state**, and ask the question the whole module is building toward — does a *quantum* coupling measure read the injected coupling $K$ as faithfully as PLV does?

**Prerequisites:** `11_kuramoto_dyad_and_ground_truth`.

**What you'll be able to do**
- Map each oscillator's phase to the **naive phase qubit** $|\psi(t)\rangle = (|0\rangle + e^{i\theta(t)}|1\rangle)/\sqrt{2}$ and time-average the joint projector into a bipartite density matrix $\rho_{AB}$ with `phase_qubit_state` and `joint_density_matrix`.
- Read $\rho_{AB}$ at a low and a high coupling and see *where* the coupling lives: not in the diagonal, but in the **off-block coherence** that grows as the dyad locks.
- Evaluate two quantum coupling measures — **quantum mutual information** (QMI) and **Bures-coupling** — beside the two classical baselines (**PLV**, **cosine correlation**) at a handful of couplings.
- Sweep $K$ across the transition region and watch all four measures climb together, so that — at first glance — the quantum measures look like they recover the ground truth in the same way as the baselines do.

In `11` you built the bench: a dyad whose coupling $K$ you set, and the classical numbers that already read it. Here you add the quantum machinery the course spent four modules assembling, and put it on the same bench. By the end you will have a working quantum coupling pipeline — phases in, a bipartite state, a coupling number out — and a sweep in which every measure, classical and quantum alike, rises with $K$. It will *look like it works*. Whether the quantum measures actually add anything the baselines did not is the question we open at the very end and answer in `13`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.capstone import (
    simulate_kuramoto_dyad,
    phase_qubit_state,
    joint_density_matrix,
    coupling_qmi,
    coupling_bures,
    plv,
    cosine_correlation,
    sweep_coupling_measures,
)

np.random.seed(42)
viz.use_course_style()

## 1. From phases to a bipartite density matrix

Module `10` gave us the bridge from a signal to a quantum state, and the simplest crossing of all is the **naive phase qubit**: send a phase $\theta(t)$ to the pure state

$$ |\psi(t)\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle + e^{i\theta(t)}|1\rangle\bigr). $$

This is a point on the *equator* of the Bloch sphere, sitting at azimuth $\theta(t)$. The amplitude is fixed at $1/\sqrt{2}$ on each basis state; all the information is in the phase, which is exactly what a Kuramoto oscillator carries. `phase_qubit_state` applies this map to a whole phase trace at once.

To couple two oscillators we form the joint state at each instant, $|\Psi(t)\rangle = |\psi_A(t)\rangle \otimes |\psi_B(t)\rangle$ — a pure product state living in the genuine **tensor product** $\mathcal{H}_A \otimes \mathcal{H}_B$. A single such instant carries no coupling at all (a product state is, by construction, uncoupled). The coupling has to emerge from the *ensemble*: we time-average the projector across the whole run,

$$ \rho_{AB} = \mathbb{E}_t\bigl[\,|\Psi(t)\rangle\langle\Psi(t)|\,\bigr], $$

which `joint_density_matrix` computes. Averaging is where the physics enters. When the two phases drift independently, every relative angle $\theta_A - \theta_B$ occurs equally often and the averaged off-block terms wash out — $\rho_{AB}$ collapses toward the maximally mixed $I_4/4$. When the phases lock, the relative angle sits near one value, those terms survive the average, and a specific **off-block coherence** appears. That surviving term is the *same* circular average $\langle e^{i(\theta_A - \theta_B)}\rangle$ that `11` identified as the heart of PLV — now sitting inside an operator.

Let us build $\rho_{AB}$ at two couplings and look at it directly: an uncoupled run ($K = 0$) and a well-locked run ($K = 0.5$, near the top of the transition region we sweep below).

In [ ]:
# A first look at the embedding: one phase trace -> a (T, 2) array of qubit amplitudes.
theta_demo, _ = simulate_kuramoto_dyad(K=0.5, duration=200.0, seed=0)
psi_demo = phase_qubit_state(theta_demo)
print("phase trace shape :", theta_demo.shape)
print("qubit-state array :", psi_demo.shape, "(rows = time, cols = |0>, |1> amplitudes)")
print("first state |psi(0)> :", psi_demo[0])  # (|0> + e^{i theta_0}|1>)/sqrt(2)

# Build the time-averaged bipartite density matrix at a low and a high coupling.
K_low, K_high = 0.0, 0.5
rho_low = joint_density_matrix(*simulate_kuramoto_dyad(K=K_low, duration=200.0, seed=0))
rho_high = joint_density_matrix(*simulate_kuramoto_dyad(K=K_high, duration=200.0, seed=0))

fig_low = viz.plot_density_matrix(
    rho_low, title=f"$\\rho_{{AB}}$ at $K = {K_low}$  (uncoupled)"
)
plt.show()

fig_high = viz.plot_density_matrix(
    rho_high, title=f"$\\rho_{{AB}}$ at $K = {K_high}$  (phase-locked)"
)
plt.show()

**Read the figure.** Two density matrices, each shown as a pair of $4 \times 4$ annotated heatmaps — the real part on the left, the imaginary part on the right — over the basis ordered $|00\rangle, |01\rangle, |10\rangle, |11\rangle$ (indices $0,1,2,3$).

At $K = 0$ (top), $\rho_{AB}$ is essentially $I_4/4$: the diagonal reads $0.25$ in all four slots and every off-diagonal entry sits at $0.00$, real and imaginary parts alike. With no coupling, every relative phase occurred equally often, the average erased the coherences, and the joint state is the maximally mixed, manifestly **product** matrix — the operator-level picture of "uncoupled."

At $K = 0.5$ (bottom), the diagonal is *unchanged* — still $0.25$ across the board. This is the right place to slow down: the coupling does **not** show up on the diagonal. Each oscillator alone still visits every phase, so each marginal stays on the Bloch equator and each diagonal block stays maximally mixed. What changes is one specific pair of off-diagonal cells — the **off-block coherence** linking $|01\rangle$ and $|10\rangle$ (the entries at row/column $1,2$). Their real part has lifted to about $0.23$ and their imaginary part to about $\pm 0.05$, while every other off-diagonal entry stays near zero. That single block is the phase-locking signal, written into the operator: the more tightly the dyad locks, the larger that off-block coherence grows. Hold onto this — it is the thread the whole capstone pulls on.

## 2. Four coupling measures, side by side

We now have a quantum object, $\rho_{AB}$, that visibly encodes the coupling. The point of building it was to feed it to a *quantum* coupling measure — and to read that measure against the classical baselines on the very same runs. Four measures stand side by side.

| Measure | What it reads | Type |
|---|---|---|
| **Quantum mutual information** $\;I(A{:}B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB})$ | how far $\rho_{AB}$ sits from the product $\rho_A \otimes \rho_B$, in nats | quantum, information-theoretic (M2) |
| **Bures-coupling** $\;d_B(\rho_{AB}, \rho_A \otimes \rho_B)$ | the transport distance from $\rho_{AB}$ to that same product | quantum, transport-geometric (M3–M4) |
| **PLV** $\;\lvert\langle e^{i(\theta_A - \theta_B)}\rangle\rvert$ | the consistency of the phase difference | classical, phase-only |
| **Cosine correlation** $\;\lvert\mathrm{corr}(\cos\theta_A, \cos\theta_B)\rvert$ | whether the raw rhythms rise and fall together | classical, Euclidean-like |

The two quantum measures both compare $\rho_{AB}$ against the product of its marginals $\rho_A \otimes \rho_B$ — the state the dyad *would* have if it were uncoupled — but they measure that gap with two different rulers: an information-theoretic one (QMI, the quantum relative entropy from Module 02) and a transport-geometric one (the Bures distance, the quantum Wasserstein (the Bures bridge) from Module 03). Both are zero exactly when $\rho_{AB} = \rho_A \otimes \rho_B$, i.e. when the dyad is uncoupled. The two classical baselines are the yardsticks from `11`. We evaluate all four at a handful of couplings spanning the transition.

In [ ]:
K_demo = [0.0, 0.1, 0.2, 0.3, 0.5]

print(f"{'K':>5}  {'QMI (nats)':>11}  {'Bures':>8}  {'PLV':>7}  {'corr':>7}")
print("-" * 46)
for K in K_demo:
    t1, t2 = simulate_kuramoto_dyad(K=K, duration=200.0, seed=0)
    rho_ab = joint_density_matrix(t1, t2)
    qmi_K = coupling_qmi(rho_ab)
    bures_K = coupling_bures(rho_ab)
    plv_K = plv(t1, t2)
    corr_K = abs(cosine_correlation(t1, t2))
    print(f"{K:>5.2f}  {qmi_K:>11.3f}  {bures_K:>8.3f}  {plv_K:>7.3f}  {corr_K:>7.3f}")

**Read the output.** Read any column from top to bottom: every one of the four climbs as $K$ grows. At $K = 0$ all four sit near their floor — QMI and Bures barely above zero (a finite noisy run never averages to a perfect product), PLV and cosine correlation likewise small. By $K = 0.5$ each has risen substantially: QMI and Bures have grown several-fold from their $K = 0$ values, while PLV and the cosine correlation have climbed toward one. The two quantum measures and the two classical ones tell the same qualitative story on these runs — *more injected coupling, larger measure* — which is the first thing we would demand of any honest coupling statistic.

They climb at *different rates* and live on *different scales* — QMI in nats with no fixed ceiling here, Bures as a distance, PLV and corr bounded in $[0,1]$ — so the raw numbers are not directly comparable across rows. What is shared, and what matters at this stage, is the direction: all four increase with the ground truth. To see the shapes rather than five sampled rows, we sweep $K$ continuously.

## 3. The $K$-sweep — all four against the ground truth

A table samples; a sweep draws the curve. `sweep_coupling_measures` runs the whole pipeline for each $K$ on a grid — simulate the dyad, embed the phases, build $\rho_{AB}$, evaluate all four measures — and returns the four curves together, so we can see how each measure responds to coupling across a continuum.

The grid deserves a word. With natural frequencies this close ($\omega_1 = 1.0$, $\omega_2 = 1.2$), the dyad locks *fast*: as `11` already hinted, most of the climb happens between $K = 0$ and $K = 0.5$, and beyond that the measures merely tighten an already-locked dyad and flatten into a plateau. A wide grid would spend most of its points on that flat plateau and show a single sharp step. So we focus the sweep on the **transition region**, $K \in [0, 0.5]$, where the measures actually change — that is where the comparison is informative, and it is the same window the next notebook builds on.

In [ ]:
# Transition-focused grid: the dyad locks by K ~ 0.5, so we sample where the
# measures actually move rather than wasting points on the post-lock plateau.
k_grid = np.linspace(0.0, 0.5, 16)
sweep = sweep_coupling_measures(k_grid, duration=200.0, seed=0)

# Report the PLV span so we can confirm the grid spreads the transition (not all-saturated).
print("grid:", k_grid.shape[0], "couplings from", f"{k_grid[0]:.2f}", "to", f"{k_grid[-1]:.2f}")
print(f"PLV spans [{sweep['plv'].min():.3f}, {sweep['plv'].max():.3f}]")

panels = [
    ("Quantum mutual information  $I(A{:}B)$  (nats)", sweep["qmi"], COLORS["quantum"]),
    ("Bures-coupling  $d_B(\\rho_{AB},\\,\\rho_A\\otimes\\rho_B)$", sweep["bures"], COLORS["source"]),
    ("Phase-locking value  PLV", sweep["plv"], COLORS["target"]),
    ("Cosine correlation  $|\\mathrm{corr}(\\cos\\theta_A,\\cos\\theta_B)|$", sweep["corr"], COLORS["flow"]),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (title, values, color) in zip(axes.ravel(), panels):
    ax.plot(sweep["K"], values, "-o", color=color, lw=2.0, markersize=4)
    ax.set_title(title, pad=8, fontsize=12)
    ax.set_ylabel("coupling measure")
for ax in axes[-1]:
    ax.set_xlabel(r"injected coupling  $K$")
fig.suptitle(
    "All four measures rise with the injected coupling $K$",
    fontsize=14, fontweight="bold", y=1.0,
)
fig.tight_layout()
plt.show()

**Read the figure.** Four panels, one per measure, each plotting the measure against the injected coupling $K$ across the transition region $[0, 0.5]$.

The two **quantum** panels sit on top. *Quantum mutual information* (top-left) starts near zero at $K = 0$ — independent oscillators give $\rho_{AB} \approx I_4/4$ with marginals $\approx I/2$, so $I(A{:}B) \approx 0$ — then climbs steeply through the transition and begins to level off as the dyad locks. *Bures-coupling* (top-right) does the same: near zero when the joint state is essentially the product of its marginals, rising as the off-block coherence we saw in §1 pushes $\rho_{AB}$ away from that product.

The two **classical** panels sit below. *PLV* (bottom-left) lifts off its small $K = 0$ floor and rises sharply toward one — across this grid it spans roughly the printed range, the spread we deliberately tuned the grid to capture. *Cosine correlation* (bottom-right) traces the same ascent from a near-zero floor.

Stand back and the four panels rhyme: every measure rises with the ground truth $K$, steeply through the transition and then easing as the dyad locks. On the strength of this figure, the quantum measures appear to recover the injected coupling exactly as the classical baselines do — the quantum machinery *looks like it works*.

## Your turn

**Warm-up.** In §1 the diagonal of $\rho_{AB}$ stayed at $0.25$ in all four slots at *both* couplings, while only the off-block $|01\rangle\langle10|$ cells changed. Explain in words why the diagonal is pinned to $0.25$ regardless of $K$ — think about what each oscillator looks like *on its own* when its phase visits the whole circle, and what that says about each single-qubit marginal. Which part of $\rho_{AB}$, then, is the only place the coupling can register?

**Go further.** The sweep above samples the transition region $K \in [0, 0.5]$. Predict what the four panels would look like on a much wider grid such as $K \in [0, 4]$, and say which qualitative feature — the steep rise or the plateau — would dominate the picture and why. Then describe, in words, what that wide-grid figure would *hide* that the transition-focused grid makes visible.

**Challenge.** Quantum mutual information here is reported in *nats*. PLV and the cosine correlation are bounded in $[0,1]$, but QMI has no such ceiling in general. Reason about what would cap the QMI in *this* construction: recall that each marginal stays near $I/2$ (the Bloch equator), and that $I(A{:}B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB})$. State the largest value QMI could reach when both marginals are exactly maximally mixed and $\rho_{AB}$ becomes pure, and explain in words why the noisy, locked dyad lands well below that ceiling.

## What you built

- You crossed the bridge from `10` with a real signal: you mapped each oscillator's phase to the **naive phase qubit** $|\psi(t)\rangle = (|0\rangle + e^{i\theta}|1\rangle)/\sqrt{2}$ and time-averaged the joint projector into a bipartite density matrix $\rho_{AB}$ — a complete *phases-in, quantum-state-out* pipeline.
- You read $\rho_{AB}$ at a low and a high coupling and located the coupling precisely: not on the diagonal (which stays maximally mixed), but in the **off-block coherence** linking $|01\rangle$ and $|10\rangle$ — the operator-level echo of the very circular average that defines PLV.
- You evaluated two **quantum** coupling measures — quantum mutual information and Bures-coupling — beside the two **classical** baselines, and saw all four rise together across a handful of couplings.
- You swept $K$ across the transition region and produced the four-panel figure in which every measure climbs with the ground truth, confirming the PLV spread the grid was chosen to capture.

Stand back and take the win: you assembled a working quantum coupling pipeline out of the machinery this course spent four modules building, put it on the honest bench from `11`, and watched it track the injected coupling. As far as this figure can tell, *it works* — the quantum measures rise with $K$ exactly as the baselines do.

## What's next

That is precisely the moment to ask the harder question. Rising with $K$ is the bar `11` set, and PLV already clears it — so do the quantum measures add anything PLV did not, or are they reading the same thing in a more elaborate language? In `13_the_redundancy_reveal` we put that question to a direct test.

## References

- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C
- Y. Kuramoto, *Chemical Oscillations, Waves, and Turbulence* (Springer, 1984). DOI:10.1007/978-3-642-69689-3
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017) — quantum mutual information $I(A{:}B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB})$. DOI:10.1017/9781316809976
- D. Bures, "An extension of Kakutani's theorem on infinite product measures to the tensor product of semifinite $w^*$-algebras", *Trans. Amer. Math. Soc.* **135**, 199–212 (1969). DOI:10.1090/S0002-9947-1969-0236719-2
- D. Trevisan, *Optimal transport methods for quantum systems* (arXiv:2202.02091, 2022). DOI:10.48550/arXiv.2202.02091

**Previous:** `notebooks/04_QuantumOptimalTransport/11_kuramoto_dyad_and_ground_truth.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/13_the_redundancy_reveal.ipynb`